#### Deep Learning


Evaluate whether neural networks can improve RUL prediction compared with
the classical ML models from Phase 3.

The first neural-network model is an MLP.

If sequence-based modeling is justified, LSTM/GRU models will be evaluated
after the MLP baseline.

In [16]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_dataset

train_df, test_df, rul_df = load_dataset("FD001")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (20631, 26)
Test shape: (13096, 26)


Load Dataset
Notebook starts from the same C-MAPSS training data used
in the previous phases.

This maintains consistency across the project.

In [2]:
RUL_CAP = 125

train_df["max_cycle"] = (
    train_df.groupby("unit")["cycle"].transform("max")
)

train_df["RUL_raw"] = (
    train_df["max_cycle"] - train_df["cycle"]
)

train_df["RUL"] = (
    train_df["RUL_raw"].clip(upper=RUL_CAP)
)

print("RUL minimum:", train_df["RUL"].min())
print("RUL maximum:", train_df["RUL"].max())
print("RUL missing:", train_df["RUL"].isna().sum())

RUL minimum: 0
RUL maximum: 125
RUL missing: 0


#### Reconstruct RUL Target

The Phase 1 RUL definition is reproduced so that the deep-learning
experiment uses the same prediction target as the classical ML models.

In [3]:
sensor_cols = [
    f"sensor_{i}"
    for i in range(1, 22)
]

diff_cols = [
    f"{sensor}_diff1"
    for sensor in sensor_cols
]

feature_cols = sensor_cols + diff_cols

print("Number of features:", len(feature_cols))

Number of features: 42


#### Define Model Features

The MLP uses the original sensor measurements together with the validated
one-cycle difference features from Phase 2.

This gives the neural network both current sensor information and recent
cycle-to-cycle change.

In [4]:
# Recreate Phase 2 difference features

train_df = train_df.sort_values(
    ["unit", "cycle"]
).reset_index(drop=True)

for sensor in sensor_cols:
    train_df[f"{sensor}_diff1"] = (
        train_df.groupby("unit")[sensor].diff(1)
    )

print("Difference features recreated:", len(diff_cols))

Difference features recreated: 21


In [5]:
print(train_df[diff_cols].head())

   sensor_1_diff1  sensor_2_diff1  sensor_3_diff1  sensor_4_diff1  \
0             NaN             NaN             NaN             NaN   
1             0.0            0.33            2.12            2.54   
2             0.0            0.20           -3.83            1.06   
3             0.0            0.00           -5.20           -2.33   
4             0.0            0.02            0.06            4.35   

   sensor_5_diff1  sensor_6_diff1  sensor_7_diff1  sensor_8_diff1  \
0             NaN             NaN             NaN             NaN   
1             0.0             0.0           -0.61           -0.02   
2             0.0             0.0            0.51            0.04   
3             0.0             0.0            0.19            0.03   
4             0.0             0.0           -0.45           -0.05   

   sensor_9_diff1  sensor_10_diff1  ...  sensor_12_diff1  sensor_13_diff1  \
0             NaN              NaN  ...              NaN              NaN   
1           -2.1

In [6]:
X = train_df[feature_cols].copy()

y = train_df["RUL"].copy()

X = X.fillna(0)

print("Missing feature values:", X.isna().sum().sum())

print("X shape:", X.shape)

print("y shape:", y.shape)

Missing feature values: 0
X shape: (20631, 42)
y shape: (20631,)


#### Handle Temporal Missing Values

The first cycle of each engine has no previous observation, so its
difference features are naturally missing.

These initial differences are represented as zero because no change from
a previous cycle is available.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

groups = train_df["unit"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

groups_train = groups.iloc[train_idx]
groups_val = groups.iloc[val_idx]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Training engines:", groups_train.nunique())
print("Validation engines:", groups_val.nunique())

Training rows: 16561
Validation rows: 4070
Training engines: 80
Validation engines: 20


#### Engine-Level Validation Split

The validation split is performed at the engine level.

This prevents observations from the same engine trajectory appearing in
both training and validation data.

The same principle was used during classical ML evaluation.

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)

Training shape: (16561, 42)
Validation shape: (4070, 42)


#### Feature Scaling

The input features are standardized using statistics learned only from
the training engines.

The same transformation is then applied to the validation data.

This prevents information from the validation set entering preprocessing.

In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


#### Initialize Deep Learning Framework

TensorFlow/Keras is used to build the first neural-network model.

The first model is intentionally simple because it serves as the bridge
between classical tabular ML and sequence models.

In [10]:
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,865 (19.00 KB)

 Trainable params: 4,865 (19.00 KB)

 Non-trainable params: 0 (0.00 B)

#### Build MLP Model

The MLP contains two hidden layers with ReLU activation.

The final layer contains one output because the task is regression:
predicting a single RUL value.

Dropout is included to reduce overfitting.

In [11]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 3877.7051 - mae: 49.3660 - val_loss: 763.9326 - val_mae: 22.1664
Epoch 2/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 719.1006 - mae: 21.5632 - val_loss: 514.7571 - val_mae: 18.2904
Epoch 3/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 631.7689 - mae: 20.2111 - val_loss: 493.7953 - val_mae: 17.9034
Epoch 4/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 603.5950 - mae: 19.7922 - val_loss: 474.4894 - val_mae: 17.5588
Epoch 5/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 588.1494 - mae: 19.5185 - val_loss: 453.4227 - val_mae: 17.2571
Epoch 6/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 563.8251 - mae: 19.1493 - val_loss: 443.7759 - val_mae: 17.0816
Epoch 7/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 548.6476 - mae: 18.8892 - val_loss: 436.5997 - val_mae: 16.7366
Epoch 8/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 536.3901 - mae: 18.6738 - val_loss: 418.6635 - val_mae: 16.5240

#### Train MLP

The MLP is trained using the engine-level training set.

Early stopping monitors validation loss and restores the best model
weights when validation performance stops improving.

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred_mlp = model.predict(
    X_val_scaled,
    verbose=0
).ravel()

mlp_mae = mean_absolute_error(y_val, y_pred_mlp)

mlp_rmse = np.sqrt(
    mean_squared_error(y_val, y_pred_mlp)
)

print("MLP MAE:", mlp_mae)
print("MLP RMSE:", mlp_rmse)

MLP MAE: 11.732884407043457
MLP RMSE: 16.16789466415135


####  MLP Benchmark

The MLP provides the deep-learning baseline for comparison with
sequence-based models.

MLP performance:

- MAE: 11.73 cycles
- RMSE: 16.17 cycles

The MLP will be used as the reference model for the LSTM experiment.



In [13]:
print("Training epochs completed:", len(history.history["loss"]))

Training epochs completed: 68


In [14]:
print(
    "Best validation loss:",
    min(history.history["val_loss"])
)

Best validation loss: 261.4008483886719


#### Training Stability

Training history was inspected to confirm that the model trained
successfully and that early stopping selected the best validation state.

In [17]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting",
        "KNN",
        "Decision Tree",
        "Linear Regression",
        "Mean Baseline",
        "MLP"
    ],
    "MAE": [
        13.621029,
        13.976901,
        14.211369,
        14.947291,
        17.712183,
        36.965226,
        mlp_mae
    ],
    "RMSE": [
        18.799942,
        18.877456,
        20.248179,
        21.141506,
        21.634332,
        41.672689,
        mlp_rmse
    ]
})

comparison = comparison.sort_values("MAE").reset_index(drop=True)

comparison

,Model,MAE,RMSE
0,MLP,11.732884,16.167895
1,Random Forest,13.621029,18.799942
2,Gradient Boosting,13.976901,18.877456
3,KNN,14.211369,20.248179
4,Decision Tree,14.947291,21.141506
5,Linear Regression,17.712183,21.634332
6,Mean Baseline,36.965226,41.672689


#### Compare Classical ML and MLP

The MLP is compared with the Phase 3 classical models using the same
evaluation metrics.

The main question is whether the neural network improves upon the
Random Forest baseline.

In [18]:
best_model = comparison.iloc[0]

print("Best model:", best_model["Model"])
print("Best MAE:", best_model["MAE"])
print("Best RMSE:", best_model["RMSE"])

Best model: MLP
Best MAE: 11.732884407043457
Best RMSE: 16.16789466415135


#### Identify Best Model

The best model is selected based on the lowest validation MAE.

A neural network is only considered an improvement if it provides better
validation performance than the existing classical baseline.

In [19]:
rf_mae = 13.621029

mlp_improvement = (
    (rf_mae - mlp_mae) / rf_mae
) * 100

print(
    "MLP improvement over Random Forest (%):",
    mlp_improvement
)

MLP improvement over Random Forest (%): 13.861982034958908


#### MLP vs Random Forest

The MLP performance was compared directly with the Phase 3 Random Forest.

A positive improvement indicates that the MLP reduced MAE.

A negative improvement indicates that Random Forest remains the stronger
model.

#### Prepare Temporal Sequences

The MLP treats each engine cycle as an independent observation.

To investigate whether historical information improves RUL prediction,
a sequence-based LSTM model will be evaluated next.

A 20-cycle window will be used so that each sample contains recent engine
history.

In [21]:
sequence_features = sensor_cols

print("Number of sequence features:", len(sequence_features))
print(sequence_features)

Number of sequence features: 21
['sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


In [ ]:
X_seq_data = train_df[sequence_features].copy()
y_seq_data = y.copy()

X_seq_data = X_seq_data.fillna(0)

print("X shape:", X_seq_data.shape)
print("y shape:", y_seq_data.shape)
print("Missing values:", X_seq_data.isna().sum().sum())

KeyError: 'RUL'